In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kews" / "__init__.py").is_file())
NOTEBOOK_DIR = REPO_ROOT / "experiments/amoc"
for path in (REPO_ROOT, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUTPUT_DIR = REPO_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from amoc_utils import get_control_all, plot_grouped_on_axis

import kews
from kews.observables import time_delay

LABEL_FONTSIZE = 16
YLABEL_FONTSIZE = 12
LEGEND_FONTSIZE = 14
TICK_FONTSIZE = 13
EWS_MEMBER_ALPHA = 0.50


### Ablating forcing

In [ ]:
# -----------------------------
# Config
# -----------------------------
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_FILE = DATA_DIR / "amoc_si_mldmax_2Juan.nc"

STATE_VARS = [
    "amoc26N_ssp245",
    "amoc48N_ssp245",
]

# Exclude Denmark Straits forcing for this ablation.
FORCING_VARS = [
    "mldmax_labr_ssp245",
    "mldmax_irm_ssp245",
    "mldmax_sco_ssp245",
    "mldmax_gin_ssp245",
]

CONTROL_PREPROCESS = "rolling_mean"   # 'none' | 'zscore' | 'rolling_mean'
CONTROL_ROLLING_WINDOW_YEARS = 10

YEAR0 = 2015
WL = 0.2

MEMBER_LABELS = np.array(list("abcdefghij"))
C1_MEMBERS = {"a", "i"}
C1_MASK = np.array([m in C1_MEMBERS for m in MEMBER_LABELS], dtype=bool)

CONTROL_METHOD = "augmented"
AUGMENTED_FUTURE_INPUT = "future"


STATE_LABELS = {
    "amoc26N_ssp245": r"AMOC $26^\circ$N",
    "amoc48N_ssp245": r"AMOC $48^\circ$N",
}

FORCING_LABELS = {
    "mldmax_labr_ssp245": "Labrador Sea MLD max",
    "mldmax_irm_ssp245": "Irminger Sea MLD max",
    "mldmax_sco_ssp245": "Scotland-Iceland Basin MLD max",
    "mldmax_gin_ssp245": "Greenland–Iceland–Norway Seas MLD max",
}

# -----------------------------
# Load data
# -----------------------------
ds = xr.open_dataset(DATA_FILE)
years = YEAR0 + ds["time"].values.astype(int)

state_dict = {}
for state_var in STATE_VARS:
    state_all = np.asarray(ds[state_var].values, dtype=np.float32)
    state_dict[state_var] = state_all

control_dict = {}
for forcing_var in FORCING_VARS:
    control_all = get_control_all(
        ds,
        forcing_var,
        preprocess=CONTROL_PREPROCESS,
        rolling_window=CONTROL_ROLLING_WINDOW_YEARS,
    )
    control_dict[forcing_var] = np.asarray(control_all, dtype=np.float32)

print("Dataset:", DATA_FILE)
print("States:", STATE_VARS)
print("Forcing ablation vars:", FORCING_VARS)
print("Control preprocess:", CONTROL_PREPROCESS, f"(window={CONTROL_ROLLING_WINDOW_YEARS} yr)")
print("Members C1:", MEMBER_LABELS[C1_MASK].tolist())
print("Members C0:", MEMBER_LABELS[~C1_MASK].tolist())

# -----------------------------
# Helpers
# -----------------------------
def compute_control_reskmd_tdf(trajectories, controls, wL,
                             control_method="augmented",
                             augmented_future_input="future"):
    out = []
    for traj, u in zip(trajectories, controls):
        data = np.asarray(traj)
        res = kews.resdmd(data, 
            window=wL, control=u, 
            observable=time_delay(n_delay=20), 
            control_method=control_method, future_input=augmented_future_input
        )
        out.append(np.asarray(res, dtype=float))
    return out


def plot_distribution(ax, years_full, series_list, color, label, start_idx=0):
    arr = np.array(series_list, dtype=float)
    arr[:, :start_idx] = np.nan
    
    t_plot = years_full[:arr.shape[1]]
    for i in range(arr.shape[0]):
        ax.plot(t_plot, arr[i], "--", color=color, linewidth=1.5, alpha=EWS_MEMBER_ALPHA)
    median = np.nanmedian(arr, axis=0)
    ax.plot(t_plot, median, "-", color=color, linewidth=2, label=label)


# Set > 0 if you want to hide the initial burn-in region.
start_idx = 0

# -----------------------------
# Compute ablation results
# -----------------------------
# Evaluate four forcings at two latitudes across ten members.
ablation_results = {}

for state_var in STATE_VARS:
    state_all = state_dict[state_var]

    for forcing_var in FORCING_VARS:
        control_all = control_dict[forcing_var]

        tipping_series = compute_control_reskmd_tdf(
            state_all[C1_MASK],
            control_all[C1_MASK],
            WL,
            control_method=CONTROL_METHOD,
            augmented_future_input=AUGMENTED_FUTURE_INPUT,
        )
        
        non_tipping_series = compute_control_reskmd_tdf(
            state_all[~C1_MASK],
            control_all[~C1_MASK],
            WL,
            control_method=CONTROL_METHOD,
            augmented_future_input=AUGMENTED_FUTURE_INPUT,
        )
        
        ablation_results[(state_var, forcing_var)] = {
            "tipping": tipping_series,
            "non_tipping": non_tipping_series,
        }


In [ ]:
# -----------------------------
# Plot
# -----------------------------
ncols = len(FORCING_VARS)
fig, axes = plt.subplots(
    3, ncols,
    figsize=(5.0 * ncols, 10.0),
    sharex="col"
)

for col, forcing_var in enumerate(FORCING_VARS):
    forcing_label = FORCING_LABELS.get(forcing_var, forcing_var)
    control_all = control_dict[forcing_var]
    
    # Row 1: forcing time series
    ax = axes[0, col]
    plot_grouped_on_axis(
        ax,
        years,
        control_all,
        C1_MASK,
        ylabel="Forcing" if col == 0 else "",
        log_scale=False,
    )
    ax.set_title(forcing_label, fontsize=LABEL_FONTSIZE)
    ax.grid(alpha=0.2)
    
    # Row 2: 26N ResKMD with control (TDF)
    ax = axes[1, col]
    res = ablation_results[("amoc26N_ssp245", forcing_var)]
    plot_distribution(ax, years, res["tipping"], "C3", "Tipping", start_idx=start_idx)
    plot_distribution(ax, years, res["non_tipping"], "C0", "Non-tipping", start_idx=start_idx)
    ax.set_ylabel(r"ResKMD with control (TDF)" + "\n" + STATE_LABELS["amoc26N_ssp245"] if col == 0 else "", fontsize=YLABEL_FONTSIZE)
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(alpha=0.2)
    
    # Row 3: 48N ResKMD with control (TDF)
    ax = axes[2, col]
    res = ablation_results[("amoc48N_ssp245", forcing_var)]
    plot_distribution(ax, years, res["tipping"], "C3", "Tipping", start_idx=start_idx)
    plot_distribution(ax, years, res["non_tipping"], "C0", "Non-tipping", start_idx=start_idx)
    ax.set_ylabel(r"ResKMD with control (TDF)" + "\n" + STATE_LABELS["amoc48N_ssp245"] if col == 0 else "", fontsize=YLABEL_FONTSIZE)
    ax.set_xlabel("Year", fontsize=LABEL_FONTSIZE)
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(alpha=0.2)

# Clean up legends: keep only one.
for ax in axes.flat:
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

axes[1, 0].legend(loc="upper left", frameon=False, fontsize=LEGEND_FONTSIZE)

for ax in axes.flat:
    ax.set_xlim(years[0], years[-1])
    ax.tick_params(labelsize=TICK_FONTSIZE)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ablate_amoc_forcing.png', dpi=300, bbox_inches='tight')
plt.show()
